# Macro Pipeline: WDI Download + Panel Build

Fetches WDI indicators, caches raw data, updates registry, and builds `macro_panel.parquet` with coverage reports.

In [1]:
from __future__ import annotations
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import requests
import hashlib
import yaml
from datetime import datetime
import wbgapi as wb

pd.options.mode.copy_on_write = True


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for p in [start] + list(start.parents):
        if (p / 'data').exists() and (p / 'notebooks').exists():
            return p
    raise ValueError('Repo root not found')

ROOT = find_repo_root()

PATHS = {
    'data_processed': ROOT / 'data' / 'processed',
    'data_raw': ROOT / 'data' / 'raw',
    'reports': ROOT / 'reports',
}


def read_parquet_pyarrow(path: str | Path) -> pd.DataFrame:
    table = pq.read_table(str(path))
    return table.to_pandas()


def write_parquet_pyarrow(df: pd.DataFrame, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, str(path))


REGISTRY_PATH = PATHS['data_raw'] / 'macro' / '_registry.yml'

def read_registry() -> list[dict]:
    if not REGISTRY_PATH.exists():
        return []
    with open(REGISTRY_PATH, 'r') as f:
        data = yaml.safe_load(f) or []
    return data


def write_registry(entries: list[dict]) -> None:
    REGISTRY_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(REGISTRY_PATH, 'w') as f:
        yaml.safe_dump(entries, f, sort_keys=False)


def update_registry(entry: dict) -> None:
    entries = read_registry()
    entries.append(entry)
    write_registry(entries)


AGGREGATES = set(wb.economy.aggregates())


In [2]:
# WDI indicator mapping
INDICATORS = {
    # Real activity / development
    'gdp_pc_real': 'NY.GDP.PCAP.KD',
    'gdp_growth': 'NY.GDP.MKTP.KD.ZG',
    'gdp_pc_growth': 'NY.GDP.PCAP.KD.ZG',
    'inv_gdp': 'NE.GDI.FTOT.ZS',
    'pop': 'SP.POP.TOTL',
    'pop_growth': 'SP.POP.GROW',

    # Prices / monetary
    'inflation_cpi': 'FP.CPI.TOTL.ZG',
    'exrate_lcu_per_usd': 'PA.NUS.FCRF',
    'credit_private_gdp': 'FS.AST.PRVT.GD.ZS',
    'money_broad_gdp': 'FM.LBL.BMNY.GD.ZS',

    # Fiscal / state size
    'gov_cons_gdp': 'NE.CON.GOVT.ZS',
    'tax_rev_gdp': 'GC.TAX.TOTL.GD.ZS',
    'debt_gdp': 'GC.DOD.TOTL.GD.ZS',

    # External sector
    'trade_gdp': 'NE.TRD.GNFS.ZS',
    'exports_gdp': 'NE.EXP.GNFS.ZS',
    'imports_gdp': 'NE.IMP.GNFS.ZS',
    'ca_gdp': 'BN.CAB.XOKA.GD.ZS',
    'natres_rents_gdp': 'NY.GDP.TOTL.RT.ZS',

    # Demography
    'urban_share': 'SP.URB.TOTL.IN.ZS',
    'dep_ratio': 'SP.POP.DPND',
}


In [3]:
# Fetch WDI with caching

def _cache_key(indicators: dict, start: int, end: int) -> str:
    payload = str(sorted(indicators.items())) + f"|{start}|{end}"
    return hashlib.sha1(payload.encode('utf-8')).hexdigest()[:12]


def fetch_wdi(indicators: dict[str,str], start: int, end: int) -> pd.DataFrame:
    key = _cache_key(indicators, start, end)
    cache_path = PATHS['data_raw'] / 'macro' / 'wdi' / f'cache_{key}.parquet'
    if cache_path.exists():
        print('Using cache:', cache_path)
        return read_parquet_pyarrow(cache_path)

    rows = []
    for var_name, code in indicators.items():
        url = f"https://api.worldbank.org/v2/country/all/indicator/{code}?format=json&per_page=20000&date={start}:{end}"
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        if not isinstance(data, list) or len(data) < 2:
            raise ValueError(f"Unexpected response for {code}")
        for item in data[1]:
            iso3 = (item.get('countryiso3code') or '').upper().strip()
            if not iso3:
                continue
            try:
                year = int(item['date'])
            except Exception:
                continue
            val = item.get('value')
            rows.append({
                'iso3': iso3,
                'year': year,
                'var_name': var_name,
                'value': val,
                'source': 'wdi',
                'indicator_code': code,
            })
        print('Fetched', var_name, code, 'rows:', len(data[1]))

    df = pd.DataFrame(rows)
    df['iso3'] = df['iso3'].str.upper().str.strip()
    df = df[df['iso3'].str.len() == 3]
    df = df[~df['iso3'].isin(AGGREGATES)]

    write_parquet_pyarrow(df, cache_path)

    # Also write a dated raw file for auditability
    dated = PATHS['data_raw'] / 'macro' / 'wdi' / f"wdi_long_{datetime.now().strftime('%Y%m%d')}.parquet"
    if not dated.exists():
        write_parquet_pyarrow(df, dated)

    # Update registry
    update_registry({
        'dataset': 'wdi',
        'pull_date': datetime.now().isoformat(timespec='seconds'),
        'start_year': start,
        'end_year': end,
        'indicators': indicators,
        'cache_path': str(cache_path),
    })

    return df


In [4]:
# Build macro panel

def build_macro_panel(wdi_long: pd.DataFrame, *, iso3_universe: set[str] | None = None) -> pd.DataFrame:
    # Pivot to wide
    wide = wdi_long.pivot_table(index=['iso3','year'], columns='var_name', values='value', aggfunc='mean').reset_index()

    # Transformations
    transforms = {
        'gdp_pc_real': 'log',
        'pop': 'log',
        'exrate_lcu_per_usd': 'log'
    }

    for var, t in transforms.items():
        if var in wide.columns:
            newcol = f"macro_{var}_{t}"
            wide[newcol] = np.log(wide[var].astype(float))

    # Rename non-transformed to macro_* prefix
    for var in [c for c in wide.columns if c not in ['iso3','year'] and not c.startswith('macro_')]:
        if var in transforms:
            continue
        wide = wide.rename(columns={var: f"macro_{var}"})

    # Coverage reports
    core_vars = [c for c in wide.columns if c.startswith('macro_')]
    cov = pd.DataFrame({
        'var': core_vars,
        'missing_share_overall': [wide[v].isna().mean() for v in core_vars],
    })
    write_parquet_pyarrow(cov, PATHS['reports'] / 'macro_coverage' / 'coverage_overall.parquet')

    if iso3_universe:
        subset = wide[wide['iso3'].isin(iso3_universe)]
        cov_sub = pd.DataFrame({
            'var': core_vars,
            'missing_share_efw_iso3': [subset[v].isna().mean() for v in core_vars],
        })
        write_parquet_pyarrow(cov_sub, PATHS['reports'] / 'macro_coverage' / 'coverage_efw_iso3.parquet')

    return wide

# Fetch WDI
wdi_long = fetch_wdi(INDICATORS, start=1970, end=2023)

# EFW iso3 universe from elections_master
path_elec = PATHS['data_processed'] / 'elections_master.parquet'
elec = read_parquet_pyarrow(path_elec)
ef_iso3 = set(elec[elec['efw_country'] == True]['iso3'].dropna().unique()) if 'efw_country' in elec.columns else None

macro_panel = build_macro_panel(wdi_long, iso3_universe=ef_iso3)

# Save macro panel
out_path = PATHS['data_processed'] / 'macro_panel.parquet'
write_parquet_pyarrow(macro_panel, out_path)
print('wrote', out_path)


Fetched gdp_pc_real NY.GDP.PCAP.KD rows: 14364


Fetched gdp_growth NY.GDP.MKTP.KD.ZG rows: 14364


Fetched gdp_pc_growth NY.GDP.PCAP.KD.ZG rows: 14364


Fetched inv_gdp NE.GDI.FTOT.ZS rows: 14364


Fetched pop SP.POP.TOTL rows: 14364


...

Fetched ca_gdp BN.CAB.XOKA.GD.ZS rows: 14364


Fetched natres_rents_gdp NY.GDP.TOTL.RT.ZS rows: 14364


Fetched urban_share SP.URB.TOTL.IN.ZS rows: 14364


Fetched dep_ratio SP.POP.DPND rows: 14364


wrote /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/macro_panel.parquet
